# 🎙️ NLP Assignment 2: Low-Resource Speech-to-Text Translation
## Irish → English Cascaded Pipeline

**Module:** Natural Language Processing (MSc AI)  
**Deadline:** 10th May 2026  
**Weighting:** 50% of module

---

This notebook will guide you through building a cascaded **Automatic Speech Recognition (ASR) → Machine Translation (MT)** pipeline for translating Irish speech into English text.

### How to use this notebook
- Cells marked `# ✅ PROVIDED` are complete — run them as-is.
- Cells marked `# 📝 YOUR CODE HERE` require you to write your own code.
- Markdown cells with 💡 are hints. Read them before coding.
- Don't skip sections — each depends on the previous.

### Recommended runtime
Go to **Runtime → Change runtime type → T4 GPU** before starting.

---

### Table of Contents
1. [Setup & Installation](#setup)
2. [Dataset Loading & Exploration](#data)
3. [Audio Preprocessing](#audio)
4. [Baseline ASR — Whisper](#asr)
5. [Machine Translation — NLLB](#mt)
6. [Full Pipeline](#pipeline)
7. [Evaluation](#eval)
8. [Error Analysis](#error)
9. [Improved System](#improved)
10. [Final Results Table](#results)

---
## Section 1: Setup & Installation <a id='setup'></a>

In [1]:
# ✅ PROVIDED — Install all required libraries
# This may take 2-3 minutes on first run.

!pip install -q transformers datasets evaluate sacrebleu jiwer librosa soundfile torchaudio
!pip install -q accelerate>=0.26.0

print('✅ Libraries installed.')

zsh:1: 0.26.0 not found
✅ Libraries installed.


In [2]:
# ✅ PROVIDED — Imports

import os
import json
import time
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch

from pathlib import Path
from IPython.display import Audio, display

from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    pipeline
)

import evaluate

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

🖥️  Device: cpu


---
## Section 2: Dataset Loading & Exploration <a id='data'></a>

The dataset is the **IWSLT 2026 Irish–English Speech Translation** data.

Repository: https://github.com/shashwatup9k/iwslt2026_ga-eng

**Dataset structure (once cloned):**
```
iwslt2026_ga-eng/
├── data/
│   ├── train/
│   │   ├── audio/          # .wav files
│   │   ├── transcripts.txt # Irish transcriptions
│   │   └── translations.txt # English reference translations
│   ├── dev/
│   └── test/
└── README.md
```

> ⚠️ **Always check the actual repository structure** — it may differ slightly from the above. Run `!find iwslt2026_ga-eng -type f | head -30` after cloning to see the real layout.

In [3]:
# ✅ PROVIDED — Clone the dataset

!git clone https://github.com/shashwatup9k/iwslt2026_ga-eng.git 2>/dev/null || echo 'Already cloned.'

# Inspect what was downloaded
print('\n📂 Repository contents:')
!find iwslt2026_ga-eng -type f | sort | head -40


📂 Repository contents:
iwslt2026_ga-eng/.git/config
iwslt2026_ga-eng/.git/description
iwslt2026_ga-eng/.git/HEAD
iwslt2026_ga-eng/.git/hooks/applypatch-msg.sample
iwslt2026_ga-eng/.git/hooks/commit-msg.sample
iwslt2026_ga-eng/.git/hooks/fsmonitor-watchman.sample
iwslt2026_ga-eng/.git/hooks/post-update.sample
iwslt2026_ga-eng/.git/hooks/pre-applypatch.sample
iwslt2026_ga-eng/.git/hooks/pre-commit.sample
iwslt2026_ga-eng/.git/hooks/pre-merge-commit.sample
iwslt2026_ga-eng/.git/hooks/pre-push.sample
iwslt2026_ga-eng/.git/hooks/pre-rebase.sample
iwslt2026_ga-eng/.git/hooks/pre-receive.sample
iwslt2026_ga-eng/.git/hooks/prepare-commit-msg.sample
iwslt2026_ga-eng/.git/hooks/push-to-checkout.sample
iwslt2026_ga-eng/.git/hooks/sendemail-validate.sample
iwslt2026_ga-eng/.git/hooks/update.sample
iwslt2026_ga-eng/.git/index
iwslt2026_ga-eng/.git/info/exclude
iwslt2026_ga-eng/.git/logs/HEAD
iwslt2026_ga-eng/.git/logs/refs/heads/main
iwslt2026_ga-eng/.git/logs/refs/remotes/origin/HEAD
iwslt2026_ga

In [ ]:
# 📝 YOUR CODE HERE
# Task: Set the correct paths to the data splits based on what you saw above.
# Adapt these paths to match the ACTUAL repository structure.

DATA_ROOT = Path('iwslt2026_ga-eng')

# TODO: Update these paths to match the actual folder structure
AUDIO_DIR = DATA_ROOT / 'data' / 'dev' / 'audio'   # <-- update if needed
TRANS_FILE = DATA_ROOT / 'data' / 'dev' / 'transcripts.txt'  # Irish transcripts
REF_FILE   = DATA_ROOT / 'data' / 'dev' / 'translations.txt' # English references

print('Audio dir exists:', AUDIO_DIR.exists())
print('Transcripts exist:', TRANS_FILE.exists())
print('References exist:', REF_FILE.exists())

In [ ]:
# 📝 YOUR CODE HERE
# Task: Load the transcripts and references into Python lists.
# Print the first 5 examples of each so you can verify they look right.

# HINT: Each line in the file corresponds to one utterance.
# Use open() and .strip() to clean whitespace.

with open(TRANS_FILE) as f:
    irish_transcripts = [line.strip() for line in f if line.strip()]

with open(REF_FILE) as f:
    english_refs = [line.strip() for line in f if line.strip()]

# TODO: Print dataset statistics
# - How many utterances are there?
# - Do the counts match?
# - What does the first example look like?

print(f'Number of transcripts: {len(irish_transcripts)}')
print(f'Number of references:  {len(english_refs)}')
print()

# TODO: Print the first 3 examples (transcript + reference pairs)
for i in range(3):
    print(f'--- Example {i+1} ---')
    print(f'Irish:   {irish_transcripts[i]}')
    print(f'English: {english_refs[i]}')
    print()

In [ ]:
# ✅ PROVIDED — Get list of audio files

audio_files = sorted(list(AUDIO_DIR.glob('*.wav')))
print(f'Found {len(audio_files)} audio files.')
if audio_files:
    print('First 5:', [f.name for f in audio_files[:5]])

---
## Section 3: Audio Inspection & Preprocessing <a id='audio'></a>

Before feeding audio to an ASR model, you need to understand its properties:
- **Sample rate** — Whisper expects 16,000 Hz (16 kHz)
- **Duration** — very short or very long clips may cause issues
- **Format** — most models expect mono-channel float arrays

💡 **Tip:** Librosa's `load()` function can resample audio automatically.

In [ ]:
# ✅ PROVIDED — Inspect one audio file

if audio_files:
    sample_file = audio_files[0]
    waveform, sr = librosa.load(sample_file, sr=None)  # sr=None preserves original rate
    
    print(f'File:        {sample_file.name}')
    print(f'Sample rate: {sr} Hz')
    print(f'Duration:    {len(waveform)/sr:.2f} seconds')
    print(f'Shape:       {waveform.shape}')
    print(f'Min/Max:     {waveform.min():.3f} / {waveform.max():.3f}')
    
    # Listen to it!
    display(Audio(waveform, rate=sr))

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a function to load and preprocess an audio file.
# Requirements:
#   - Resample to 16000 Hz (Whisper's required rate)
#   - Return a float32 numpy array
#   - Handle mono audio only

TARGET_SR = 16000

def load_audio(filepath, target_sr=TARGET_SR):
    """
    Load an audio file and resample to target_sr.
    Returns: numpy array (float32), sample rate
    """
    # TODO: Implement this function
    # HINT: librosa.load(filepath, sr=target_sr) will handle resampling
    raise NotImplementedError('Implement load_audio()')

# Test your function
if audio_files:
    wav, sr = load_audio(audio_files[0])
    assert sr == TARGET_SR, f'Sample rate should be {TARGET_SR}, got {sr}'
    assert wav.dtype == np.float32, 'Array should be float32'
    print(f'✅ load_audio works. Duration: {len(wav)/sr:.2f}s')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Compute and report basic statistics for the audio dataset.
# Report:
#   - Total number of audio files
#   - Average, min, max duration in seconds
#   - Total hours of audio

# TODO: Loop over audio_files, load each one, collect durations
# HINT: Use a sample of 50 files if the full set is large, for speed.

sample_files = audio_files[:50]  # Use subset if dataset is large
durations = []

for f in sample_files:
    # TODO: Load file, compute duration in seconds, append to durations
    pass

# TODO: Print statistics
print('=== Audio Statistics ===')
# print(f'Files sampled: ...')
# print(f'Average duration: ... s')
# etc.

---
## Section 4: Baseline ASR — Whisper <a id='asr'></a>

**Whisper** (Radford et al., 2022) is a multilingual ASR model from OpenAI, trained on 680,000 hours of multilingual audio. It supports Irish and requires 16 kHz mono audio input.

### Available Whisper model sizes:

| Model | Parameters | Recommended for |
|-------|-----------|----------------|
| `whisper-tiny` | 39M | Quick testing |
| `whisper-base` | 74M | Baseline (fast) |
| `whisper-small` | 244M | Better quality |
| `whisper-medium` | 769M | Good baseline |
| `whisper-large-v3` | 1.5B | Strongest (slow on CPU) |

💡 **Recommendation for baseline:** Start with `openai/whisper-small` for a reasonable speed/quality tradeoff on Colab.

In [ ]:
# 📝 YOUR CODE HERE
# Task: Load a Whisper model and processor.
# 
# Requirements:
#   - Choose a Whisper model appropriate for your baseline
#   - Load the processor and model
#   - Move model to DEVICE
#   - Justify your choice in a comment

# TODO: Choose a model ID
ASR_MODEL_ID = 'openai/whisper-small'  # Change this to your chosen model

print(f'Loading ASR model: {ASR_MODEL_ID}...')

# TODO: Load WhisperProcessor and WhisperForConditionalGeneration
# whisper_processor = ...
# whisper_model = ...

# TODO: Move model to DEVICE and set to eval mode
# whisper_model = whisper_model.to(DEVICE).eval()

# Uncomment when implemented:
# print(f'✅ Loaded {ASR_MODEL_ID}')
# print(f'   Parameters: {sum(p.numel() for p in whisper_model.parameters()):,}')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a function to transcribe a single audio file using Whisper.
#
# Steps:
#   1. Load and preprocess audio (use your load_audio function)
#   2. Create input features with the Whisper processor
#   3. Generate transcription tokens (force Irish language: forced_decoder_ids)
#   4. Decode the output tokens to text
#   5. Return the transcription string

def transcribe_with_whisper(audio_path, processor, model, language='ga'):
    """
    Transcribe a single audio file using Whisper.
    
    Args:
        audio_path: Path to .wav file
        processor:  WhisperProcessor
        model:      WhisperForConditionalGeneration
        language:   ISO code for Irish ('ga')
    Returns:
        Transcription string
    """
    # TODO: Implement transcription
    # HINT: processor.get_decoder_prompt_ids(language=language, task='transcribe')
    # HINT: inputs = processor(waveform, sampling_rate=16000, return_tensors='pt')
    # HINT: model.generate(**inputs, forced_decoder_ids=forced_ids)
    # HINT: processor.batch_decode(tokens, skip_special_tokens=True)
    raise NotImplementedError('Implement transcribe_with_whisper()')

# Test on one file
if audio_files:
    result = transcribe_with_whisper(audio_files[0], whisper_processor, whisper_model)
    print(f'Test transcription: {result}')
    print(f'Reference (Irish):  {irish_transcripts[0]}')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Run ASR on a subset of the development/test set.
#
# - Process the first MAX_SAMPLES audio files (start with 50 for speed)
# - Store transcriptions in a list
# - Track and print average processing time per file
#
# 💡 TIP: Use tqdm for a progress bar:
#   from tqdm import tqdm

from tqdm import tqdm

MAX_SAMPLES = 50  # Increase to full set when confident your code works

asr_hypotheses = []  # List of transcriptions
asr_times = []

# TODO: Loop over audio_files[:MAX_SAMPLES]
# - Time each transcription call
# - Append result to asr_hypotheses
# - Handle errors gracefully (try/except, append empty string on failure)

print(f'Total ASR hypotheses: {len(asr_hypotheses)}')
print(f'Average time per file: {np.mean(asr_times):.2f}s')

---
## Section 5: Machine Translation — NLLB <a id='mt'></a>

**NLLB-200** (No Language Left Behind, Meta AI, 2022) is a multilingual MT model covering 200 languages, including Irish (`gle_Latn`).

### Available NLLB variants:

| Model | Size | Notes |
|-------|------|-------|
| `facebook/nllb-200-distilled-600M` | 600M | Recommended for baseline |
| `facebook/nllb-200-distilled-1.3B` | 1.3B | Better quality, slower |
| `facebook/nllb-200-1.3B` | 1.3B | Non-distilled |

💡 **Language codes:** Irish = `gle_Latn`, English = `eng_Latn`

In [ ]:
# 📝 YOUR CODE HERE
# Task: Load the NLLB-200 model and tokeniser.

MT_MODEL_ID = 'facebook/nllb-200-distilled-600M'  # You may change this

SRC_LANG = 'gle_Latn'  # Irish
TGT_LANG = 'eng_Latn'  # English

print(f'Loading MT model: {MT_MODEL_ID}...')

# TODO: Load AutoTokenizer and AutoModelForSeq2SeqLM
# HINT: Use tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_ID)
# HINT: Use model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_ID)

# mt_tokenizer = ...
# mt_model = ...
# mt_model = mt_model.to(DEVICE).eval()

# print(f'✅ Loaded {MT_MODEL_ID}')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a function to translate Irish text to English using NLLB.

def translate_with_nllb(text, tokenizer, model, src_lang=SRC_LANG, tgt_lang=TGT_LANG, max_new_tokens=256):
    """
    Translate Irish text to English.
    Returns: translated string
    """
    if not text or not text.strip():
        return ''  # Handle empty input
    
    # TODO: Implement translation
    # HINT: tokenizer(text, return_tensors='pt', src_lang=src_lang)
    # HINT: forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]
    # HINT: model.generate(**inputs, forced_bos_token_id=..., max_new_tokens=...)
    # HINT: tokenizer.batch_decode(outputs, skip_special_tokens=True)
    raise NotImplementedError('Implement translate_with_nllb()')

# Test it!
test_irish = 'Dia duit, conas atá tú?'  # 'Hello, how are you?'
translation = translate_with_nllb(test_irish, mt_tokenizer, mt_model)
print(f'Irish:   {test_irish}')
print(f'English: {translation}')

---
## Section 6: Full Cascaded Pipeline <a id='pipeline'></a>

Now combine ASR → MT into one end-to-end function.

```
Audio file
    │
    ▼ load_audio()
Waveform (16kHz)
    │
    ▼ transcribe_with_whisper()
Irish transcript (text)
    │
    ▼ translate_with_nllb()
English translation (text)
```

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write the full pipeline function.

def speech_to_text_translate(audio_path,
                              asr_processor, asr_model,
                              mt_tokenizer, mt_model):
    """
    Full cascaded speech translation: audio file → English text.
    Returns: (irish_transcript, english_translation)
    """
    # TODO: Call transcribe_with_whisper, then translate_with_nllb
    # Return both the intermediate transcript and the final translation
    raise NotImplementedError('Implement speech_to_text_translate()')

# Test on a sample
if audio_files:
    transcript, translation = speech_to_text_translate(
        audio_files[0], whisper_processor, whisper_model, mt_tokenizer, mt_model
    )
    print(f'ASR Transcript: {transcript}')
    print(f'MT Translation: {translation}')
    print(f'Reference:      {english_refs[0]}')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Run the full pipeline on MAX_SAMPLES files.
# Save: transcripts, translations, and any failed indices.

baseline_transcripts = []
baseline_translations = []
failed_indices = []

# TODO: Loop, run pipeline, collect results
# Handle errors — if a file fails, store '' and record the index

print(f'Processed: {len(baseline_translations)} files')
print(f'Failed: {len(failed_indices)} files')

In [ ]:
# ✅ PROVIDED — Save baseline outputs

os.makedirs('results', exist_ok=True)

with open('results/baseline_hypotheses.txt', 'w') as f:
    for line in baseline_translations:
        f.write(line + '\n')

with open('results/references.txt', 'w') as f:
    for line in english_refs[:MAX_SAMPLES]:
        f.write(line + '\n')

print('✅ Saved results/baseline_hypotheses.txt')
print('✅ Saved results/references.txt')

---
## Section 7: Evaluation <a id='eval'></a>

We evaluate using two standard metrics:

| Metric | Description | Why use it? |
|--------|-------------|-------------|
| **BLEU** | N-gram overlap between hypothesis and reference | Standard MT metric |
| **chrF++** | Character + word n-gram F-score | More robust for morphologically rich languages |

💡 Always use **SacreBLEU** for reproducible BLEU scores — never implement BLEU from scratch.

In [ ]:
# ✅ PROVIDED — Evaluation helper functions

import sacrebleu

def compute_bleu(hypotheses, references):
    """
    Compute corpus BLEU using SacreBLEU.
    Args:
        hypotheses: list of predicted strings
        references: list of reference strings
    Returns: BLEU score (float)
    """
    # SacreBLEU expects references wrapped in a list of lists
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score

def compute_chrf(hypotheses, references):
    """
    Compute chrF++ using SacreBLEU.
    """
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    return chrf.score

def compute_coverage(hypotheses):
    """Proportion of non-empty outputs."""
    n_empty = sum(1 for h in hypotheses if not h.strip())
    return 1.0 - (n_empty / max(len(hypotheses), 1))

def compute_repetition_rate(hypotheses, threshold=0.3):
    """Proportion of outputs where the same word appears too frequently."""
    flagged = 0
    for h in hypotheses:
        words = h.split()
        if len(words) < 4:
            continue
        from collections import Counter
        freqs = Counter(words)
        if freqs.most_common(1)[0][1] / len(words) > threshold:
            flagged += 1
    return flagged / max(len(hypotheses), 1)

print('✅ Evaluation functions loaded.')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Evaluate your baseline system.
# - Filter out any failed samples (where hypothesis or reference is empty)
# - Compute BLEU, chrF++, coverage, repetition rate
# - Print a summary table

refs_subset = english_refs[:MAX_SAMPLES]

# TODO: Filter empty hypotheses / references
# HINT: zip hypotheses and refs, skip pairs where either is empty

valid_hyps = []
valid_refs = []

# TODO: Fill valid_hyps and valid_refs

# TODO: Compute metrics
bleu = 0.0  # Replace with actual computation
chrf = 0.0  # Replace with actual computation
coverage = 0.0
rep_rate = 0.0

print('=== Baseline System Results ===')
print(f'Samples evaluated:   {len(valid_hyps)}')
print(f'BLEU:                {bleu:.2f}')
print(f'chrF++:              {chrf:.2f}')
print(f'Coverage:            {coverage:.1%}')
print(f'Repetition rate:     {rep_rate:.1%}')

---
## Section 8: Error Analysis <a id='error'></a>

Quantitative metrics tell you *how much* a system gets wrong — error analysis tells you *why*.

### Common error types to look for:

| Error Type | Description | Example |
|-----------|-------------|--------|
| **ASR substitution** | Wrong word transcribed | "sráid" → "sraith" |
| **ASR deletion** | Missing words | "tá sé ann" → "sé ann" |
| **ASR hallucination** | Model confabulates plausible-sounding but wrong text | |
| **MT untranslated** | Irish word passed through to English output | |
| **MT mistranslation** | Wrong meaning | "leabhar" → "book" ✅ vs "leabhar" → "library" ❌ |
| **MT over-generation** | Unexplained extra text in output | |
| **Error propagation** | ASR error causes MT error | |

💡 Aim for a systematic analysis of at least **20 examples**.

In [ ]:
# ✅ PROVIDED — Side-by-side comparison helper

def show_examples(n=10, start=0):
    """Display examples in a readable format for error analysis."""
    for i in range(start, min(start + n, len(baseline_translations))):
        print(f'--- Example {i+1} ---')
        print(f'Irish transcript (ASR): {baseline_transcripts[i]}')
        print(f'Irish reference:        {irish_transcripts[i]}')
        print(f'English translation:    {baseline_translations[i]}')
        print(f'English reference:      {english_refs[i]}')
        print()

show_examples(n=5)

In [ ]:
# 📝 YOUR CODE HERE
# Task: Manually annotate 20+ examples with error categories.
# 
# Create a list of dictionaries, one per example, containing:
#   - 'id': example index
#   - 'asr_errors': list of error types in ASR output
#   - 'mt_errors': list of error types in MT output
#   - 'notes': your comment on the example

# Example annotation:
error_annotations = [
    {
        'id': 0,
        'asr_errors': ['substitution'],  # Fill in for real
        'mt_errors': ['none'],
        'notes': 'ASR got one word wrong but MT recovered adequately.'
    },
    # TODO: Add 19+ more annotations based on actual examples
]

# TODO: Write a simple summary of error type frequencies
from collections import Counter

all_asr_errors = [e for ann in error_annotations for e in ann['asr_errors']]
all_mt_errors  = [e for ann in error_annotations for e in ann['mt_errors']]

print('ASR error types:', Counter(all_asr_errors))
print('MT error types: ', Counter(all_mt_errors))

---
## Section 9: Improved System <a id='improved'></a>

You must implement **at least one meaningful improvement** over your baseline.

### Possible directions (choose one or more):

| Direction | Description | Effort |
|-----------|-------------|--------|
| 🔄 **Different ASR model** | Try `whisper-medium` or a Wav2Vec2 Irish model | Low |
| 🔄 **Different MT model** | Try `nllb-1.3B` or `Helsinki-NLP/opus-mt-ga-en` | Low |
| 🔧 **ASR language forcing** | Ensure Whisper transcribes in Irish, not English | Low |
| 🔧 **Beam search tuning** | Increase `num_beams` for MT | Low |
| ✂️ **Output filtering** | Remove empty/degenerate outputs before MT | Medium |
| 🧹 **Text normalisation** | Clean ASR output before MT (punctuation, case) | Medium |
| 🎛️ **Audio augmentation** | Normalise audio volume before ASR | Medium |

You must **justify your choice** in your report and **compare results** against the baseline.

In [ ]:
# 📝 YOUR CODE HERE
# Task: Implement your improved system.
#
# You may:
#   - Modify the transcription or translation functions
#   - Load a different model
#   - Add preprocessing/postprocessing steps
#
# Store your improved translations in: improved_translations

# TODO: Implement your improvement here

improved_translations = []  # Fill this with your improved system's outputs

# TODO: Evaluate the improved system with the same metrics
# improved_bleu = ...
# improved_chrf = ...

---
## Section 10: Final Results Table <a id='results'></a>

Your report must include a results table. Build it here.

In [ ]:
# 📝 YOUR CODE HERE
# Task: Build and display a results table comparing baseline and improved system.

results = pd.DataFrame([
    {
        'System': f'Baseline ({ASR_MODEL_ID} + {MT_MODEL_ID})',
        'BLEU': bleu,
        'chrF++': chrf,
        'Coverage': f'{coverage:.1%}',
        'Rep. Rate': f'{rep_rate:.1%}',
    },
    {
        'System': 'Improved (describe your system)',  # TODO: Update description
        'BLEU': 0.0,    # TODO: Replace with real values
        'chrF++': 0.0,  # TODO: Replace with real values
        'Coverage': 'N/A',
        'Rep. Rate': 'N/A',
    }
])

results = results.set_index('System')
print('=== Final Results ===')
display(results)

# Save to file
results.to_csv('results/final_results.csv')
print('\nSaved to results/final_results.csv')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a brief (3-5 sentence) interpretation of your results in this cell.
# This will help you draft the Results section of your report.

# TODO: Fill in your analysis
print("""
=== Results Interpretation ===

[Write 3-5 sentences here interpreting your results. For example:]
- How do BLEU/chrF++ compare between systems?
- Which errors are most common and why?
- Was the improvement meaningful? What might explain it?
- What are the key limitations of your approach?
""")

---
## ✅ Submission Checklist

Before submitting, confirm:

- [ ] All cells run top-to-bottom without errors
- [ ] `results/baseline_hypotheses.txt` saved
- [ ] `results/final_results.csv` saved
- [ ] Error analysis covers at least 20 examples
- [ ] Improved system is distinct from baseline and justified
- [ ] README.md explains how to run your notebook
- [ ] ACL-format report is complete (6–8 pages)
- [ ] All AI tool usage is declared in the report appendix
- [ ] All random seeds are set and documented

**Good luck! 🍀**